1. Write a CUDA C/C++ program to perform element-wise addi on of two
vectors.                                        
C[i]=A[i]+B[i]

Given: Vector size: N = 1024

In [4]:
%%writefile addvector.cu
#include<stdio.h>
#include <cuda_runtime.h>
#define N 1024

__global__ void add(int *a, int *b, int *c){
int index= blockIdx.x * blockDim.x + threadIdx.x;
c[index] = a[index] + b[index];
};
int main(void)
{
int *a, *b, *c;
int *d_a, *d_b, *d_c;
int size = N * sizeof(int);
a = (int *)malloc(size);
b = (int *)malloc(size);
c = (int *)malloc(size);
for (int i = 0; i < N; i++) {
a[i] = i * 1;
printf("%d\n",a[i]);
}
for (int i = 0; i < N; i++) {
b[i] = i * 2;
printf("%d\n",b[i]);
}

cudaMalloc((void **)&d_a, size);
cudaMalloc((void **)&d_b, size);
cudaMalloc((void **)&d_c, size);

cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

int threads = 8;
int blocks = (N + threads - 1) / threads;

add<<<blocks,threads>>>(d_a, d_b, d_c);

// Copy result back to host
cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);
printf("Array values of C:\n");
for(int i=0; i<N; i++)
printf("%d ", c[i]);

// Cleanup
cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
return 0;
}

Overwriting addvector.cu


In [5]:
!nvcc -arch=sm_75 addvector.cu -o add


In [6]:
!./add

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

2. Perform the same vector addi on as in Q1 using Thrust library only.

In [7]:
%%writefile vecadd.cu

#include<iostream>
#include<thrust/device_vector.h>
#include<thrust/host_vector.h>
#include<thrust/transform.h>
#include<thrust/functional.h>

#define N 1024

int main(){
    thrust::host_vector<int> h_a(N);
    thrust::host_vector<int> h_b(N);

    for(int i=0;i<N ;i++){
        h_a[i] = i;
        h_b[i] = 2*i;
    }

    thrust::device_vector<int> d_a = h_a;
    thrust::device_vector<int> d_b = h_b;
    thrust::device_vector<int> d_c(N);

    thrust::transform(d_a.begin(),d_a.end(),d_b.begin(),d_c.begin(),thrust::plus<int>());

    thrust::host_vector<int> h_c = d_c;

    for(int i=0;i<N;i++){
        std::cout<<h_c[i]<<" ";
    }
}

Writing vecadd.cu


In [8]:
!nvcc -arch=sm_75 vecadd.cu -o add


In [9]:
!./add

0 3 6 9 12 15 18 21 24 27 30 33 36 39 42 45 48 51 54 57 60 63 66 69 72 75 78 81 84 87 90 93 96 99 102 105 108 111 114 117 120 123 126 129 132 135 138 141 144 147 150 153 156 159 162 165 168 171 174 177 180 183 186 189 192 195 198 201 204 207 210 213 216 219 222 225 228 231 234 237 240 243 246 249 252 255 258 261 264 267 270 273 276 279 282 285 288 291 294 297 300 303 306 309 312 315 318 321 324 327 330 333 336 339 342 345 348 351 354 357 360 363 366 369 372 375 378 381 384 387 390 393 396 399 402 405 408 411 414 417 420 423 426 429 432 435 438 441 444 447 450 453 456 459 462 465 468 471 474 477 480 483 486 489 492 495 498 501 504 507 510 513 516 519 522 525 528 531 534 537 540 543 546 549 552 555 558 561 564 567 570 573 576 579 582 585 588 591 594 597 600 603 606 609 612 615 618 621 624 627 630 633 636 639 642 645 648 651 654 657 660 663 666 669 672 675 678 681 684 687 690 693 696 699 702 705 708 711 714 717 720 723 726 729 732 735 738 741 744 747 750 753 756 759 762 765 768 771 774 77

3. Compute the dot product of two vectors of size, N =1024:
Result=∑A[i]×B[i]

using Thrust and compare its performance with that on CPU.

In [10]:
%%writefile vecmu_simple.cu

#include <iostream>
#include <thrust/device_vector.h>
#include <thrust/host_vector.h>
#include <thrust/inner_product.h>

#define N 1024

int main() {
    thrust::host_vector<int> h_a(N);
    thrust::host_vector<int> h_b(N);

    for(int i = 0; i < N; i++){
        h_a[i] = i;
        h_b[i] = 2 * i;
    }

    float cpu_result = 0;
    for(int i = 0; i < N; i++){
        cpu_result += h_a[i] * h_b[i];
    }

    std::cout << "CPU Result: " << cpu_result << std::endl;

    thrust::device_vector<int> d_a = h_a;
    thrust::device_vector<int> d_b = h_b;

    float gpu_result = thrust::inner_product(
                        d_a.begin(),
                        d_a.end(),
                        d_b.begin(),
                        0.0f);

    std::cout << "GPU Result: " << gpu_result << std::endl;

    return 0;
}

Writing vecmu_simple.cu


In [11]:
!nvcc -arch=sm_75 vecmu_simple.cu -o multiply


In [12]:
!./multiply

CPU Result: 7.14779e+08
GPU Result: 7.1478e+08


4. Write a CUDA kernel for matrix mul plica on: C=A×B where Matrix size is 16
X 16. Explain why matrix mul plica on needs more computa on than
addi on

In [13]:
%%writefile matrix_mul.cu

#include <iostream>
#include <cuda.h>

#define N 16
__global__ void matrixMul(int *A, int *B, int *C) {

    int row = threadIdx.y;
    int col = threadIdx.x;

    int sum = 0;

    for(int k = 0;k<N;k++) {
        sum += A[row*N+k]*B[k*N+col];
    }

    C[row*N+col] = sum;
}

int main() {

    int h_A[N][N], h_B[N][N], h_C[N][N];

    // Initialize matrices
    for(int i = 0; i < N; i++){
        for(int j=0;j<N;j++){
            h_A[i][j] = 1;
            h_B[i][j] = 1;
        }
    }

    int *d_A, *d_B, *d_C;

    cudaMalloc(&d_A, N*N*sizeof(int));
    cudaMalloc(&d_B, N*N*sizeof(int));
    cudaMalloc(&d_C, N*N*sizeof(int));

    cudaMemcpy(d_A, h_A, N*N*sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, N*N*sizeof(int), cudaMemcpyHostToDevice);
    dim3 threadsPerBlock(N, N);
    matrixMul<<<1, threadsPerBlock>>>(d_A, d_B, d_C);

    cudaMemcpy(h_C, d_C, N*N*sizeof(int), cudaMemcpyDeviceToHost);

    std::cout << "Result Matrix C:\n";
    for(int i=0;i<N;i++){
        for(int j=0;j<N;j++){
            std::cout << h_C[i][j] << " ";
        }
        std::cout << std::endl;
    }

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Writing matrix_mul.cu


In [14]:
!nvcc -arch=sm_75 matrix_mul.cu -o mul


In [15]:
!./mul

Result Matrix C:
16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 
16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 
16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 
16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 
16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 
16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 
16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 
16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 
16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 
16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 
16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 
16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 
16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 
16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 
16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 
16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 16 


Write a CUDA C++ program using the Thrust library to compute the sum of
all elements in a vector stored on the GPU. The vector is of size 10 and it
should be ini alized with values 1,…..10.  

In [16]:
%%writefile thrust_sum.cu

#include <iostream>
#include <thrust/device_vector.h>
#include <thrust/reduce.h>

#define N 10

int main() {

    // Create device vector of size 10
    thrust::device_vector<int> d_vec(N);

    // Initialize with values 1 to 10
    for(int i = 0; i < N; i++){
        d_vec[i] = i + 1;
    }

    // Compute sum using thrust::reduce
    int sum = thrust::reduce(d_vec.begin(), d_vec.end(), 0);

    std::cout << "Sum of elements: " << sum << std::endl;

    return 0;
}

Writing thrust_sum.cu


In [17]:
!nvcc -arch=sm_75 thrust_sum.cu -o sum

In [18]:
!./sum

Sum of elements: 55


7. Write a CUDA C++ program using Thrust to sort (ascending) a vector of
integers on the GPU. Consider vector size 8 with following values: 7, 2, 9, 1,
5, 3, 8, 4. Print the vector before and a er sor ng.

In [19]:
%%writefile thrust_sort.cu

#include <iostream>
#include <thrust/device_vector.h>
#include <thrust/sort.h>

using namespace std;

int main() {

    thrust::device_vector<int> d_vec{7, 2, 9, 1, 5, 3, 8, 4};

    cout << "Before Sorting: ";
    for(int i = 0; i < d_vec.size(); i++)
        cout << d_vec[i] << " ";

    cout << endl;

    // Sort in ascending order
    thrust::sort(d_vec.begin(), d_vec.end());

    // Print after sorting
    cout << "After Sorting: ";
    for(int i = 0; i < d_vec.size(); i++)
        cout << d_vec[i] << " ";

    cout << endl;

    return 0;
}

Writing thrust_sort.cu


In [20]:
!nvcc -arch=sm_75 thrust_sort.cu -o sort

In [21]:
!./sort

Before Sorting: 7 2 9 1 5 3 8 4 
After Sorting: 1 2 3 4 5 7 8 9 
